# Arbeitspaket (AP) 1: Management & Nutzung relationaler Daten

### Persönliche Angaben (bitte ergänzen) und Prüfungsregeln

<table>
  <tr><td>Vorname:</td><td>Ruzica</td></tr>
  <tr><td>Nachname:</td><td>Barbaric</td></tr>
  <tr><td>Immatrikulationsnummer:</td><td>1111</td></tr>
  <tr><td>Modul:</td><td>Data Science</td></tr>
  <tr><td>Prüfungsdatum / Raum / Zeit:</td><td>[25.1.2026] / Raum: [2] / [10.00]</td></tr>
  <tr><td>Erlaubte Hilfsmittel:</td><td>Es gelten die Prüfungsregeln (PDF-Dokument auf EXAM-Moodle).</td></tr>
  <tr><td>Nicht erlaubt:</td><td>Es gelten die Prüfungsregeln (PDF-Dokument auf EXAM-Moodle).</td></tr>
  <tr><td>Technische Probleme:</td><td>Wenden Sie sich bei technischen Problemen umgehend an die Prüfungsaufsicht.</td></tr>
</table>

## Bewertungskriterien

**(max. erreichbare Punkte: 40)**

<table>
  <thead>
    <tr><th>Teil</th><th>Thema</th><th>Aufgaben</th><th>Punkte</th></tr>
  </thead>
  <tbody>
    <tr><td>A</td><td>Konzeptfragen: Datenmodellierung &amp; Design</td><td>A1–A3</td><td>12</td></tr>
    <tr><td>B</td><td>SQL-Fehleranalyse (cinema.db)</td><td>B1–B2</td><td>16</td></tr>
    <tr><td>C</td><td>SQL-Abfragen schreiben (cinema.db)</td><td>C1–C3</td><td>12</td></tr>
    <tr><td><b>Total</b></td><td></td><td><b>8 Aufgaben</b></td><td><b>40</b></td></tr>
  </tbody>
</table>

### Bewertungsschema

Jede Aufgabe wird für sich und **als Ganzes** bewertet. Ihre Punktzahl steht in der Überschrift.

<table>
  <thead>
    <tr><th>Kategorie</th><th>Beschreibung</th><th>Bewertung</th><th>4 Pkt</th><th>8 Pkt</th></tr>
  </thead>
  <tbody>
    <tr><td>Korrekt</td><td>Vollständig und fachlich einwandfrei: Antwort bzw. Begründung korrekt und nachvollziehbar; bei SQL-Aufgaben liefert die SQL-Abfrage das richtige Ergebnis</td><td>100%</td><td>4</td><td>8</td></tr>
    <tr><td>Minimale Mängel</td><td>Kleine Fehler (z.B. falsche Sortierung, fehlende Rundung) oder Begründung ungenau bzw. unvollständig</td><td>75%</td><td>3</td><td>6</td></tr>
    <tr><td>Mittlere Mängel</td><td>Grössere Fehler (z.B. fehlende Spalten, falsche Filterung) oder Begründung fehlt bzw. ist fachlich falsch</td><td>50%</td><td>2</td><td>4</td></tr>
    <tr><td>Gravierende Mängel</td><td>Problem bzw. Ansatz erkennbar, aber Lösung oder Schlussfolgerung im Kern falsch (z.B. Problem richtig benannt, aber falsch behoben; im Code: ungeeigneter Lösungsweg wie falscher JOIN-Typ oder fehlende Tabelle)</td><td>25%</td><td>1</td><td>2</td></tr>
    <tr><td>Nicht lauffähig / nicht sinnvoll</td><td>Antwort fehlt, Code nicht lauffähig (z.B. Syntaxfehler) oder kein Bezug zur Aufgabe</td><td>0%</td><td>0</td><td>0</td></tr>
  </tbody>
</table>

Treffen mehrere Mängel zusammen, zählt der schwerste. Alternative, fachlich korrekte Lösungswege werden ebenfalls als korrekt gewertet.

## Hinweise zur Datenbank

**Datenbank:** PostgreSQL — Datenbank `postgres`, Benutzer `pgadmin`. Die Verbindung wird im Setup-Abschnitt aufgebaut und getestet.

**Backup:** `cinema.db` (SQLite) im gleichen Verzeichnis wie dieses Notebook — nur zu verwenden, falls die PostgreSQL-Verbindung nicht funktioniert.

**Tabellen:** `movies`, `cinemas`, `staff`, `screenings`, `customers`, `tickets`

Das ER-Diagramm und die Tabellenbeschreibungen finden Sie im Sheet **«ER-Diagram»** bzw. **«Description»** der beiliegenden Datei `cinema_data.xlsx`.

**Antworten auf Konzeptfragen** tragen Sie bitte direkt in die vorgesehenen Markdown-Zellen ein.

## Setup (wird nicht bewertet)

In [2]:
# Libraries
import os
import sqlite3
import pandas as pd
from sqlalchemy import create_engine, text

# Settings
import warnings
warnings.filterwarnings('ignore')

print('Working directory:', os.getcwd())

Working directory: /workspaces/DS_HS2026_LNW/DS_HS2026_LNW_I_Examples/AP01


### Datenbankverbindung zu PostgreSQL

In [3]:
# Zugangsdaten
user     = 'pgadmin'
password = 'geheim'
port     = '5432'
database = 'postgres'

# Host ermitteln: im Devcontainer heisst der PostgreSQL-Service 'db',
# bei Docker-in-Docker ist er unter 'localhost' erreichbar.
host = None
for candidate in ('db', 'localhost'):
    try:
        probe = create_engine(f'postgresql://{user}:{password}@{candidate}:{port}/{database}',
                              connect_args={'connect_timeout': 3})
        with probe.connect() as connection:
            connection.execute(text('SELECT 1'))
        probe.dispose()
        host = candidate
        break
    except Exception:
        pass

# Connection URL für alle folgenden Abfragen
db_connection_url = f'postgresql://{user}:{password}@{host}:{port}/{database}'

# Test der Datenbankverbindung
engine = create_engine(db_connection_url)
with engine.connect() as connection:
    print('Host:      ', host)
    print('Datenbank: ', connection.execute(text('SELECT current_database()')).scalar())
engine.dispose()

Host:       localhost
Datenbank:  postgres


### Daten nach PostgreSQL laden

In [4]:
# Tabellen aus der SQLite-Datei cinema.db nach PostgreSQL übertragen
tables = ['movies', 'cinemas', 'staff', 'screenings', 'customers', 'tickets']

sqlite_engine = create_engine('sqlite:///cinema.db')
engine = create_engine(db_connection_url)

for table in tables:
    df = pd.read_sql_query(f'SELECT * FROM {table}', con=sqlite_engine)
    df.to_sql(table, engine, if_exists='replace', index=False)
    print(f'{table:<12}{len(df):>5} Zeilen')

sqlite_engine.dispose()
engine.dispose()

movies         15 Zeilen
cinemas         6 Zeilen
staff          30 Zeilen
screenings    300 Zeilen
customers     505 Zeilen
tickets      2000 Zeilen


> **Hinweis — Backup mit SQLite**
>
> Falls die PostgreSQL-Verbindung nicht funktioniert, verwenden Sie für alle Aufgaben die
> SQLite-Datenbank `cinema.db`. Ersetzen Sie dazu in den Code-Zellen:
> ```python
> engine = create_engine(db_connection_url)
> ```
> durch:
>
> ```python
> engine = create_engine('sqlite:///cinema.db')
> ```
>
> Tabellen und Inhalte sind identisch, und alle Abfragen dieser Prüfung laufen unverändert
> in beiden Datenbanken.

---

<div style="background-color:#2E4057; color:white; padding:16px 24px; border-radius:6px; font-size:18px; margin-top:8px;">
&#9654;&nbsp; <strong>Ab hier beginnt die bewertete Prüfung — 40 Punkte</strong><br>
<span style="font-size:15px; font-weight:normal;">Tragen Sie Ihre Antworten direkt in die vorgesehenen Zellen ein und führen Sie alle Code-Zellen aus.</span>
</div>

---
## Teil A – Konzeptfragen (12 Punkte)

*Antworten ohne Bezug zum Schema von `cinema.db` werden nicht gewertet.*

### A1 (4 Punkte)

Die Tabelle `screenings` enthält `base_price` und die Tabelle `tickets` enthält `paid_price`.

Ein Datenbankarchitekt sagt: *«`paid_price` ist redundant — der Preis steht bereits in `screenings.base_price`. Die Spalte sollte entfernt werden.»*

Hat er recht? Begründen Sie Ihre Antwort mit einem konkreten Szenario aus dem Kinobetrieb (z.B. Rabattaktionen, Gutscheine).

**Ihre Antwort:**

> Die Spalte sollte nicht entfernt werden, da die paid_price Kolonne sich aus der Tabelle tickets auf die Beziehung des Kunden zum Ticketpreis bezieht, denn auf der customer Tabelle ist ersichlich, dass einige Kunden (Spalte membership_type) ein Standard, Premium, Senior oder Student Ticket haben, was sich auf den base_price von Tabelle screenings auswirken kann. Somit kann man genauer feststellen was der konkrete Umsatz sein würde.

KORRIGIERT
Der Kernunterschied fehlt. Der Punkt, der die Frage entscheidet, ist: Die beiden Spalten speichern zwei verschiedene Sachverhalte.
base_price = Listenpreis der Vorstellung, gleich für alle Tickets dieser Vorstellung
paid_price = tatsächlich bezahlter Betrag pro Ticket
Darum ist paid_price keine Redundanz. Redundanz wäre es nur, wenn beide immer gleich wären.

Antwort (Nein) →
Fachbegriff (keine Redundanz, weil anderer Sachverhalt) → 
Unterschied der beiden Spalten
→ konkretes Szenario (Rabatt/Mitgliedschaft) 
→ Folge beim Entfernen (Umsatz nicht berechenbar)

### A2 (4 Punkte)

Das Datenteam analysiert die Tabelle `customers` und findet folgende Werte in der Spalte `membership_type`:

```
Standard, standard, STANDARD, Standart, Premium, premium, Student, student, Senior
```

Beschreiben Sie das konkrete Datenproblem und nennen Sie die passende Data-Quality-Eigenschaft aus den Vorlesungsfolien (z.B. accurate, consistent, complete, reliable).

**Ihre Antwort:**

> Hierbei handelt es sich um Data Quality Management, genauer Consitent. Die Data Remediation wurde entweder noch nicht gemacht oder nicht sauber gemacht. Da die gleichen Begriffe unterschiedlich geschrieben wurden (Grossbuchstaben vs Kleinbuchstaben) können Abfragen je nach Programmiersprache unterschiedliche Ergebnisse liefern. Beispielsweise könnte dies zur Folge haben, dass bei einem GROUP BY land ein Standard oder standard nicht gezählt wird, da nur nach STANDARD gesucht wird. Also würde die Auswertung falsch sein.

KORREKTUR
Tippfehler Standart fehlt
land gibt es in dieser Aufgabe garnicht, sondern membership_type!!!
bei GROUP BY geht es um gruppen, nicht um zählen. Es würden also mehrere Gruppen gebildet und nicht nur die richtigen. 

### A3 (4 Punkte)

In `cinema.db` ist `paid_price` bei keinem Ticket höher als der `base_price` der zugehörigen Vorstellung. Angenommen, ein neues Ticket hätte `paid_price` = CHF 18.00 bei einem `base_price` von CHF 16.00: Wäre das **zwingend** ein Datenfehler? Begründen Sie in einem Satz.

**Ihre Antwort:**


Es muss nicht zwingend ein Datenfehler sein, da es als Geschäftsfall denkbar wäre. Beispielsweise könnte dies eine Special Edition sein, wo das Kino einen Event veranstaltet hat und einen Aufpreis dafür verlangte oder es könnten Sitze der VIP Lounge sein o.ä. Daher kann man das so pauschal nicht als Datenfehler abtun.
Der Wert weicht vom Muster aller bisherigen Tickets ab. Man sollte ihn deshalb prüfen (gegen die Preisregeln des Kinos), bevor man ihn als gültig oder als Fehler einstuft.

---
## Teil B – SQL-Fehleranalyse (16 Punkte)

*Jede Abfrage läuft fehlerfrei durch, liefert aber ein falsches oder irreführendes Geschäftsergebnis. Die tatsächlichen Ergebnisse aus `cinema.db` sind angegeben.*

### B1 (8 Punkte)

**Ziel:** Anzahl verkaufter Tickets pro Kino — für **alle** Kinos in der Datenbank.

```sql
SELECT c.name, COUNT(*) AS ticket_count
FROM screenings s
JOIN tickets t ON s.screeningid = t.screeningid
JOIN cinemas c ON s.cinemaid = c.cinemaid
GROUP BY c.name;
```

Die Tabelle `cinemas` enthält **6 Kinos**. Tatsächliches Ergebnis der SQL-Abfrage: nur **5 Zeilen**, Ticketanzahl zwischen 350 und 450.

Das Management fragt: *«Zeig mir auch Kinos, die noch keine Tickets verkauft haben.»*

**Aufgabe:** Schreiben Sie die korrigierte SQL-Abfrage, sodass **alle 6 Kinos** erscheinen und ein Kino ohne Tickets korrekt mit `0` ausgewiesen wird. Begründen Sie jede Änderung gegenüber der ursprünglichen SQL-Abfrage.

**Begründung der Änderungen:**

Es muss ein LEFT JOIN statt JOIN angewendet werden, damit auch die Kinos angezeigt werden, die keine Tickets verkauft haben. 
1. Fehler JOIN	Aurora verschwindet ganz, weil es keinen Partner in screenings hat → nur 5 Zeilen
2. Fehler COUNT(*)	Nachdem man auf LEFT JOIN umgestellt hat, erscheint Aurora mit 1 statt 0, weil die NULL-Zeile mitgezählt wird

Warum FROM cinemas? Weil LEFT JOIN alle Zeilen der linken Tabelle behält, also der hinter FROM. Wir wollen alle Kinos, also muss cinemas links stehen.
Warum beide JOINs LEFT? Wäre nur der erste LEFT und der zweite ein normaler JOIN, würde Aurora beim zweiten JOIN wieder herausfallen, weil es auch in tickets keinen Partner hat.

In [11]:
db_connection_url = f'postgresql://{user}:{password}@{host}:{port}/{database}'

engine = create_engine(db_connection_url)
df_q1=pd.read_sql_query('''
SELECT 
c.name, COUNT(t.ticketid) AS ticket_count
FROM cinemas c
LEFT JOIN screenings s ON s.cinemaid = c.cinemaid
LEFT JOIN tickets t ON s.screeningid = t.screeningid
GROUP BY c.name
ORDER BY ticket_count DESC
''', con=engine)
engine.dispose()
df_q1

,name,ticket_count
0,Kinok,441
1,Pathé Küchlin,436
2,Rex Kino,386
3,Cineplexx,383
4,Arthouse Metropol,354
5,Kino Aurora,0


### B2 (8 Punkte)

**Ziel:** Liste aller Kunden (`customerid`, `firstname`, `lastname`), die noch nie ein Ticket gekauft haben.

```sql
SELECT c.customerid, c.firstname, c.lastname
FROM customers c
JOIN tickets t ON c.customerid = t.customerid
WHERE t.ticketid IS NULL;
```

Tatsächliches Ergebnis: 0 Zeilen — obwohl es Kunden ohne Tickets gibt.

**Aufgabe:** Schreiben Sie die korrigierte SQL-Abfrage, die alle Kunden ohne Ticket ausgibt. Begründen Sie jede Änderung gegenüber der ursprünglichen SQL-Abfrage.

**Begründung der Änderungen:**

Der INNER JOIN behält nur Kunden, die mindestens ein Ticket haben. Kunden ohne Ticket sind danach schon weg, und für die übrigen ist t.ticketid nie NULL. Darum findet WHERE t.ticketid IS NULL nichts. Mit LEFT JOIN bleiben alle Kunden erhalten. Bei Kunden ohne Ticket sind die Spalten aus tickets NULL, und genau diese filtert WHERE t.ticketid IS NULL heraus.

**Korrigierte SQL-Abfrage:**

In [21]:
engine = create_engine(db_connection_url)

df_B2 = pd.read_sql_query("""
SELECT 
c.customerid,
c.firstname, 
c.lastname
FROM customers c 
LEFT JOIN tickets t ON c.customerid = t.customerid
WHERE t.ticketid IS NULL;
""", con=engine)

engine.dispose()
df_B2

,customerid,firstname,lastname
0,CU0503,Julia,Schmid
1,CU0505,Sandra,Fischer
2,CU0045,Sara,Schmidt
3,CU0504,Kevin,Keller
4,CU0432,Lucas,Wolf
5,CU0259,Max,Keller
6,CU0019,David,Krüger
7,CU0502,Thomas,Berger
8,CU0334,Mia,Krüger
9,CU0270,Marie,Braun


---
## Teil C – SQL-Abfragen schreiben (12 Punkte)

Schreiben Sie SQL-Abfragen auf `cinema.db` und begründen Sie Ihre Vorgehensweise jeweils in 1–2 Sätzen. Worauf die Begründung eingehen soll, steht bei der Aufgabe.

### C1 (4 Punkte)

Die gültigen Werte in `customers.membership_type` sind `Standard`, `Premium`, `Student` und `Senior`. Schreiben Sie eine SQL-Abfrage, die alle Kunden (`customerid`, `firstname`, `lastname`, `membership_type`) mit einem ungültigen Wert in `membership_type` ausgibt — d.h. alle Werte ausser diesen vier. Begründen Sie Ihre Filterbedingung.

**Begründung der Vorgehensweise:**

Alle informationen sind in der Tabelle customers auffindbar. Daher benötigt es keinen Join. mit dieser Filterweise, kann mit genau dieser exakten Schreibweise ausgeschlossen werden. SQL achtet nicht auf gross kleinschreibung ausser es sind werte in ''. Durch dieses Ausschliessungsverfahren bleiben nur noch ungültige Werte wie 'Standart' 'standard' oder 'STANDARD'.

**SQL-Abfrage:**

In [49]:
engine = create_engine(db_connection_url)

df_C1 = pd.read_sql_query("""
SELECT membership_type, customerid,firstname,lastname
FROM customers
WHERE membership_type NOT IN ('Standard', 'Premium', 'Student', 'Senior')
ORDER BY customerid
""", con=engine)

engine.dispose()
df_C1

,membership_type,customerid,firstname,lastname
0,standard,CU0501,Anna,Müller
1,STANDARD,CU0502,Thomas,Berger
2,Standart,CU0503,Julia,Schmid
3,premium,CU0504,Kevin,Keller
4,student,CU0505,Sandra,Fischer


### C2 (4 Punkte)

Wie hoch ist der durchschnittliche bezahlte Ticketpreis (`paid_price`) pro Mitgliedschaftstyp (`membership_type`)? Geben Sie `membership_type` und den Durchschnittspreis (`avg_price`) aus, gerundet auf 2 Dezimalstellen mit `ROUND(CAST(... AS NUMERIC), 2)`. Sortieren Sie absteigend nach `avg_price`. Begründen Sie, welche Tabellen Sie verbinden müssen und warum.

**Begründung der Vorgehensweise:**

Ich muss die Tabellen customer und tickets verbinden, weil auf der Tabelle customer die Informationen der membership_types stehen aber auf der tickets Tabelle die paid price. daher muss ich beide verbinden und hier möchte ich nur Werte, die auch effektiv genutzt wurden. Kunden ohne Ticketkäufe können wegfallen, nicht wie beim LEFT oder Full Join. es kommen nur Paare in Frage, die zusammenpassen. Daher werden beide tabellen über den Primary Key der customer customerid und den Foreign Key der tickets customerid verbunden. anschliessend wird gruppiert und sortiert .

**SQL-Abfrage:**

In [50]:
engine = create_engine(db_connection_url)

df_C2 = pd.read_sql_query("""
SELECT c.membership_type, ROUND(CAST(AVG(paid_price)AS NUMERIC), 2) AS avg_price
FROM tickets t
JOIN customers c ON c.customerid=t.customerid
GROUP BY c.membership_type
ORDER BY avg_price DESC
""", con=engine)

engine.dispose()
df_C2

,membership_type,avg_price
0,Premium,15.62
1,Student,15.59
2,Standard,15.49
3,Senior,15.34


### C3 (4 Punkte)

Welche Kinos haben mehr als 400 Tickets verkauft? Geben Sie `name` und Anzahl Tickets (`ticket_count`) aus, absteigend sortiert nach `ticket_count`. Begründen Sie, welche Tabellen Sie verbinden müssen und warum.

**Begründung der Vorgehensweise:**

die Tabellen cinemas, screenings und tickets müssen miteinander verbunden werden denn man kann nicht direkt die kinonamen mit den Ticketpreisen verbinden. daher muss man den Umweg machen:
cinemas= c.name
screenings = Verbindung zu cinemas und tickets
ticket = ticketid

da wir die Anzahl Tickets (in Tabelle tickets) pro Kino (in Tabelle cinemas) suchen, müssen wir so vorgehen. Hierfür ein Join, da wir nur Wertpaare haben möchten, die sich überschneiden. ich will keine Kinos, die keine Tickets verkauft haben. Gefiltert wird das ganze dann mit HAVING >400, da es in den Gruppen gefiltert wird. Die Bedingung bezieht sich auf ein Aggregat (COUNT). Aggregate sind im WHERE nicht erlaubt, weil WHERE einzelne Zeilen vor der Gruppierung filtert.

**SQL-Abfrage:**

In [51]:
engine = create_engine(db_connection_url)

df_C3 = pd.read_sql_query("""
SELECT c.cinemaid, c.name, COUNT(t.ticketid) AS ticket_count
FROM cinemas c
JOIN screenings s ON s.cinemaid = c.cinemaid
JOIN tickets t ON s.screeningid=t.screeningid
GROUP BY c.cinemaid,c.name
HAVING COUNT(t.ticketid)>400
ORDER BY ticket_count DESC
""", con=engine)

engine.dispose()
df_C3

,cinemaid,name,ticket_count
0,C005,Kinok,441
1,C002,Pathé Küchlin,436


---
### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [52]:
import os, platform, socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('Python Version:', python_version())
print('IP Address:', socket.gethostbyname(socket.gethostname()))
print('-----------------------------------')

-----------------------------------
POSIX
Linux | 6.8.0-1064-azure
Datetime: 2026-11-01 11:51:30
Python Version: 3.12.11
IP Address: 127.0.0.1
-----------------------------------
